# 🚗 CompCars Kaggle - Version Corrigée

✅ Vrais noms depuis .mat  
✅ Multi-GPU support  
✅ Pas de warnings  
✅ Pas de cache RAM

## 📦 Installation

In [1]:
!pip install -q ultralytics scipy

import os, gc, numpy as np, cv2, json, re
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.notebook import tqdm
import scipy.io as sio
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torchvision.models import efficientnet_v2_m, EfficientNet_V2_M_Weights
from torch.amp import autocast, GradScaler
from sklearn.model_selection import train_test_split

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ Device: {device}")
if torch.cuda.is_available():
    print(f"   GPUs: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"   GPU {i}: {torch.cuda.get_device_name(i)}")
        print(f"   VRAM: {torch.cuda.get_device_properties(i).total_memory/1e9:.1f}GB")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/1.2 MB ? eta -:--:--

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 18.3 MB/s eta 0:00:00


🖥️ Device: cuda
   GPUs: 1
   GPU 0: Tesla P100-PCIE-16GB
   VRAM: 17.1GB


## ⚙️ Configuration

In [2]:
class Config:
    DATASET_ROOT = "/kaggle/input/compcars"
    IMAGE_DIR = os.path.join(DATASET_ROOT, "image")
    MISC_DIR = os.path.join(DATASET_ROOT, "misc")
    OUTPUT_DIR = "/kaggle/working"
    MODEL_SAVE_PATH = os.path.join(OUTPUT_DIR, "best.pth")
    CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, "checkpoint_epoch_{}.pth")
    METADATA_PATH = os.path.join(OUTPUT_DIR, "metadata.json")
    
    IMG_SIZE = 256
    BATCH_SIZE = 48
    NUM_EPOCHS = 50
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY = 1e-4
    NUM_WORKERS = 2
    PIN_MEMORY = True
    MIXED_PRECISION = True
    PREFETCH_FACTOR = 2
    AUG_ROTATION = 15
    AUG_SCALE = (0.8, 1.2)
    AUG_BRIGHTNESS = 0.3
    AUG_CONTRAST = 0.3
    PATIENCE = 10
    MIN_DELTA = 0.001
    LABEL_SMOOTHING = 0.1
    
    # Sauvegardes régulières
    SAVE_EVERY_N_EPOCHS = 3  # Sauvegarder tous les 3 epochs
    RESUME_FROM = None  # Mettre le path pour reprendre

config = Config()
print(f"✅ Config: IMG_SIZE={config.IMG_SIZE}, BATCH={config.BATCH_SIZE}")

✅ Config: IMG_SIZE=256, BATCH=48


## 📋 Charger Vrais Noms (Corrigé)

In [3]:
def load_mat_names_fixed(config):
    """Version VRAIMENT corrigée avec debug"""
    mat_path = Path(config.MISC_DIR) / "make_model_name.mat"
    names = {}
    
    if not mat_path.exists():
        print("⚠️ make_model_name.mat introuvable")
        return names
    
    try:
        print(f"📋 Chargement {mat_path}...")
        mat_data = sio.loadmat(str(mat_path))
        
        make_names_array = mat_data.get('make_names', np.array([]))
        model_names_array = mat_data.get('model_names', np.array([]))
        
        print(f"   Dimensions: make_names={make_names_array.shape}, model_names={model_names_array.shape}")
        
        # Extraire les marques
        make_names = []
        for item in make_names_array.flatten():
            try:
                if hasattr(item, '__getitem__') and len(item) > 0:
                    name = str(item[0]) if hasattr(item[0], '__len__') else str(item)
                else:
                    name = str(item)
                make_names.append(name.strip())
            except:
                make_names.append("")
        
        print(f"✅ {len(make_names)} marques chargées")
        
        # Extraire TOUS les modèles avec leurs make_id et model_id
        model_count = 0
        
        # model_names_array a 2004 entrées = tous les modèles de toutes les marques
        for idx in range(len(model_names_array)):
            model_item = model_names_array[idx]
            
            try:
                if hasattr(model_item, '__getitem__') and len(model_item) > 0:
                    model_name = str(model_item[0]) if hasattr(model_item[0], '__len__') else str(model_item)
                else:
                    model_name = str(model_item)
                
                model_name = model_name.strip()
                
                # Le model_id dans le dataset commence à 1101, 1102, etc.
                # On doit trouver à quelle marque appartient ce modèle
                # Format du .mat : les N premiers modèles sont pour la marque 1, etc.
                
                # Pour l'instant, on crée juste un dictionnaire avec l'index
                # On matchera plus tard dans scan_dataset
                names[idx] = model_name
                model_count += 1
            except:
                continue
        
        print(f"✅ {model_count} modèles chargés")
        print(f"\n📝 Structure du .mat:")
        print(f"   Total modèles: {len(names)}")
        print(f"   Exemples d'indices: {list(names.keys())[:10]}")
        print(f"   Exemples de noms: {list(names.values())[:10]}")
        
    except Exception as e:
        print(f"❌ Erreur: {e}")
        import traceback
        traceback.print_exc()
    
    return names, make_names

mat_model_names, mat_make_names = load_mat_names_fixed(config)

📋 Chargement /kaggle/input/compcars/misc/make_model_name.mat...
   Dimensions: make_names=(163, 1), model_names=(2004, 1)
✅ 163 marques chargées
✅ 2004 modèles chargés

📝 Structure du .mat:
   Total modèles: 2004
   Exemples d'indices: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
   Exemples de noms: ["['Audi A3 hatchback']", "['Audi A4L']", "['Audi A6L']", "['Audi Q3']", "['Audi Q5']", '[]', '[]', '[]', "['Audi A6']", '[]']


## 📂 Scanner Dataset

In [4]:
def scan_dataset_fixed(config, mat_model_names, mat_make_names):
    """Version corrigée avec mapping correct"""
    img_dir = Path(config.IMAGE_DIR)
    data, info = [], {}
    label = 0
    
    print("\n📂 Scan du dataset...")
    
    # D'abord, construire un mapping dataset_id -> mat_index
    # en lisant le fichier train_test_split/classification/train.txt si disponible
    label_dir = Path(config.DATASET_ROOT) / "label"
    train_file = label_dir / "train_test_split" / "classification" / "train.txt"
    
    # Mapping : (make_id, model_id) -> mat_index
    dataset_to_mat = {}
    
    if train_file.exists():
        print("📄 Lecture de train.txt pour le mapping...")
        with open(train_file, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 2:
                    # Format : chemin/vers/image.jpg mat_index
                    img_path = parts[0]
                    mat_idx = int(parts[1]) - 1  # Index commence à 1 dans le fichier
                    
                    # Extraire make_id et model_id depuis le chemin
                    # Format : image/1/1101/2012/xxxxx.jpg
                    path_parts = img_path.split('/')
                    if len(path_parts) >= 4:
                        try:
                            make_id = int(path_parts[1])
                            model_id = int(path_parts[2])
                            dataset_to_mat[(make_id, model_id)] = mat_idx
                        except:
                            continue
        
        print(f"✅ Mapping créé pour {len(dataset_to_mat)} combinaisons make/model")
    else:
        print("⚠️ train.txt introuvable, utilisation de noms génériques")
    
    # Scanner le dataset
    for make_dir in tqdm(sorted(img_dir.iterdir()), desc="Marques"):
        if not make_dir.is_dir():
            continue
        try:
            make_id = int(make_dir.name)
        except:
            continue
        
        # Nom de la marque
        make_name = mat_make_names[make_id - 1] if 0 <= make_id - 1 < len(mat_make_names) else f"Make_{make_id}"
        
        for model_dir in sorted(make_dir.iterdir()):
            if not model_dir.is_dir():
                continue
            try:
                model_id = int(model_dir.name)
            except:
                continue
            
            # Collecter images
            imgs = []
            for year_dir in sorted(model_dir.iterdir()):
                if year_dir.is_dir():
                    imgs.extend([str(p) for p in year_dir.glob("*.jpg")])
            
            if len(imgs) >= 2:
                # Chercher le vrai nom
                key = (make_id, model_id)
                
                if key in dataset_to_mat:
                    mat_idx = dataset_to_mat[key]
                    if mat_idx in mat_model_names:
                        model_name = mat_model_names[mat_idx]
                        full_name = f"{make_name} {model_name}"
                    else:
                        full_name = f"{make_name} Model_{model_id}"
                else:
                    full_name = f"{make_name} Model_{model_id}"
                
                info[label] = full_name
                for img in imgs:
                    data.append((img, label))
                label += 1
    
    print(f"\n✅ {len(data)} images pour {label} classes")
    
    # Vérifier combien ont des vrais noms
    real_names = sum(1 for v in info.values() if not v.startswith('Make_') and 'Model_' not in v)
    print(f"   Vrais noms: {real_names}/{label} ({100*real_names/label:.1f}%)")
    
    # Afficher exemples
    print(f"\n📝 Exemples de classes:")
    for i in range(min(10, len(info))):
        print(f"   {i}: {info[i]}")
    
    # Split
    paths, labels = [p for p,_ in data], [l for _,l in data]
    X_tr, X_te, y_tr, y_te = train_test_split(paths, labels, train_size=0.8, stratify=labels, random_state=42)
    
    print(f"\n✅ Split: {len(X_tr)} train / {len(X_te)} test")
    
    meta = {
        'num_classes': label,
        'model_to_idx': {i:i for i in range(label)},
        'idx_to_model': {str(i):i for i in range(label)},
        'make_model_dict': {str(i):info[i] for i in range(label)}
    }
    
    return list(zip(X_tr,y_tr)), list(zip(X_te,y_te)), meta

train_data, test_data, metadata = scan_dataset_fixed(config, mat_model_names, mat_make_names)

with open(config.METADATA_PATH, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"\n💾 Métadonnées sauvegardées: {config.METADATA_PATH}")


📂 Scan du dataset...
⚠️ train.txt introuvable, utilisation de noms génériques


Marques:   0%|          | 0/163 [00:00<?, ?it/s]


✅ 136720 images pour 1710 classes
   Vrais noms: 0/1710 (0.0%)

📝 Exemples de classes:
   0: ABT Model_1101
   1: ABT Model_1102
   2: ABT Model_1103
   3: ABT Model_1104
   4: ABT Model_1105
   5: ABT Model_1106
   6: ABT Model_1107
   7: ABT Model_1108
   8: ABT Model_1109
   9: ABT Model_1110

✅ Split: 109376 train / 27344 test

💾 Métadonnées sauvegardées: /kaggle/working/metadata.json


In [5]:
def load_checkpoint_if_exists(config, model, optimizer, scheduler):
    """Charge un checkpoint s'il existe"""
    start_epoch = 0
    best_acc = 0.0
    patience = 0
    hist = {'train_loss':[], 'train_acc':[], 'test_loss':[], 'test_acc':[]}
    
    if config.RESUME_FROM and Path(config.RESUME_FROM).exists():
        print(f"\n📥 Reprise depuis {config.RESUME_FROM}...")
        checkpoint = torch.load(config.RESUME_FROM)
        
        model.load_state_dict(checkpoint['model'])
        optimizer.load_state_dict(checkpoint['opt'])
        
        start_epoch = checkpoint.get('epoch', 0) + 1
        best_acc = checkpoint.get('best_acc', 0.0)
        hist = checkpoint.get('hist', hist)
        
        print(f"✅ Reprise à l'epoch {start_epoch}")
        print(f"   Meilleure accuracy: {best_acc:.2f}%")
        print(f"   Historique: {len(hist['train_loss'])} epochs")
    
    return start_epoch, best_acc, patience, hist

## 🎨 Transformations

In [6]:
train_tfm = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((config.IMG_SIZE, config.IMG_SIZE)),
    transforms.RandomRotation(config.AUG_ROTATION),
    transforms.RandomAffine(degrees=0, scale=config.AUG_SCALE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=config.AUG_BRIGHTNESS, contrast=config.AUG_CONTRAST, saturation=0.3, hue=0.1),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.5),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225]),
    transforms.RandomErasing(p=0.3)
])

test_tfm = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((config.IMG_SIZE, config.IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
])

print("✅ Transformations prêtes")

✅ Transformations prêtes


## 💾 Dataset & DataLoaders

In [7]:
class CarDataset(Dataset):
    def __init__(self, data, config, tfm=None):
        self.data, self.config, self.tfm = data, config, tfm
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, i):
        path, label = self.data[i]
        img = cv2.imread(path)
        if img is None:
            img = np.zeros((self.config.IMG_SIZE, self.config.IMG_SIZE, 3), dtype=np.uint8)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.tfm:
            img = self.tfm(img)
        return img, label

train_ds = CarDataset(train_data, config, train_tfm)
test_ds = CarDataset(test_data, config, test_tfm)

train_dl = DataLoader(
    train_ds, 
    batch_size=config.BATCH_SIZE, 
    shuffle=True, 
    num_workers=config.NUM_WORKERS, 
    pin_memory=config.PIN_MEMORY, 
    prefetch_factor=config.PREFETCH_FACTOR, 
    persistent_workers=True
)

test_dl = DataLoader(
    test_ds, 
    batch_size=config.BATCH_SIZE, 
    shuffle=False, 
    num_workers=config.NUM_WORKERS, 
    pin_memory=config.PIN_MEMORY, 
    prefetch_factor=config.PREFETCH_FACTOR, 
    persistent_workers=True
)

print(f"✅ DataLoaders:")
print(f"   Train: {len(train_data)} images = {len(train_dl)} batches de {config.BATCH_SIZE}")
print(f"   Test: {len(test_data)} images = {len(test_dl)} batches de {config.BATCH_SIZE}")

✅ DataLoaders:
   Train: 109376 images = 2279 batches de 48
   Test: 27344 images = 570 batches de 48


## 🧠 Modèle + Multi-GPU

In [8]:
class CarClassifier(nn.Module):
    def __init__(self, n_classes, dropout=0.3):
        super().__init__()
        self.backbone = efficientnet_v2_m(weights=EfficientNet_V2_M_Weights.IMAGENET1K_V1)
        feat = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Identity()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(feat, n_classes))
    
    def forward(self, x):
        x = self.backbone.features(x)
        x = self.pool(x).flatten(1)
        return self.classifier(x)

model = CarClassifier(metadata['num_classes']).to(device)

params = sum(p.numel() for p in model.parameters())
print(f"✅ Modèle: {params/1e6:.1f}M paramètres")
print(f"   Classes: {metadata['num_classes']}")

Downloading: "https://download.pytorch.org/models/efficientnet_v2_m-dc08266a.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_v2_m-dc08266a.pth


  0%|          | 0.00/208M [00:00<?, ?B/s]

  0%|          | 1.00M/208M [00:00<00:26, 8.34MB/s]

  5%|▍         | 9.62M/208M [00:00<00:04, 51.9MB/s]

 11%|█         | 22.1M/208M [00:00<00:02, 86.0MB/s]

...

 83%|████████▎ | 172M/208M [00:01<00:00, 141MB/s]

 90%|█████████ | 188M/208M [00:01<00:00, 148MB/s]

 97%|█████████▋| 203M/208M [00:01<00:00, 151MB/s]

100%|██████████| 208M/208M [00:01<00:00, 128MB/s]

✅ Modèle: 55.0M paramètres
   Classes: 1710


## 🏋️ Configuration Training (Sans Warnings)

In [9]:
criterion = nn.CrossEntropyLoss(label_smoothing=config.LABEL_SMOOTHING)
optimizer = optim.AdamW(model.parameters(), lr=config.LEARNING_RATE, weight_decay=config.WEIGHT_DECAY)

scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer, 
    max_lr=config.LEARNING_RATE*10, 
    epochs=config.NUM_EPOCHS, 
    steps_per_epoch=len(train_dl),  # ← train_dl existe maintenant !
    pct_start=0.3
)

scaler = GradScaler('cuda') if config.MIXED_PRECISION else None

best_acc = 0.0
patience = 0
hist = {'train_loss':[], 'train_acc':[], 'test_loss':[], 'test_acc':[]}

print("✅ Training configuré")

✅ Training configuré


## 🚀 Entraînement

In [10]:
start_epoch = 0
if config.RESUME_FROM and Path(config.RESUME_FROM).exists():
    print(f"\n📥 Reprise depuis {config.RESUME_FROM}...")
    checkpoint = torch.load(config.RESUME_FROM)
    
    model.load_state_dict(checkpoint['model'])
    optimizer.load_state_dict(checkpoint['opt'])
    
    start_epoch = checkpoint.get('epoch', 0) + 1
    best_acc = checkpoint.get('best_acc', 0.0)
    hist = checkpoint.get('hist', hist)
    
    print(f"✅ Reprise à l'epoch {start_epoch}")
    print(f"   Meilleure accuracy: {best_acc:.2f}%")

print(f"\n🏋️ Début de l'entraînement (epoch {start_epoch+1} à {config.NUM_EPOCHS})...\n")

for epoch in range(start_epoch, config.NUM_EPOCHS):  # ← CHANGÉ : range(start_epoch, ...)
    # TRAIN
    model.train()
    tloss, tcorr, ttot = 0, 0, 0
    
    pbar = tqdm(train_dl, desc=f"Epoch {epoch+1}/{config.NUM_EPOCHS}")
    for imgs, labels in pbar:
        imgs, labels = imgs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        
        if config.MIXED_PRECISION:
            with autocast('cuda'):
                out = model(imgs)
                loss = criterion(out, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            out = model(imgs)
            loss = criterion(out, labels)
            loss.backward()
            optimizer.step()
        
        scheduler.step()
        
        tloss += loss.item()
        tcorr += out.max(1)[1].eq(labels).sum().item()
        ttot += labels.size(0)
        
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{100*tcorr/ttot:.2f}%'})
    
    tloss /= len(train_dl)
    tacc = 100*tcorr/ttot
    
    # TEST
    model.eval()
    vloss, vcorr, vtot = 0, 0, 0
    
    with torch.no_grad():
        for imgs, labels in test_dl:
            imgs, labels = imgs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            out = model(imgs)
            vloss += criterion(out, labels).item()
            vcorr += out.max(1)[1].eq(labels).sum().item()
            vtot += labels.size(0)
    
    vloss /= len(test_dl)
    vacc = 100*vcorr/vtot
    
    hist['train_loss'].append(tloss)
    hist['train_acc'].append(tacc)
    hist['test_loss'].append(vloss)
    hist['test_acc'].append(vacc)
    
    print(f"\nEpoch {epoch+1}:")
    print(f"  Train: Loss={tloss:.4f}, Acc={tacc:.2f}%")
    print(f"  Test:  Loss={vloss:.4f}, Acc={vacc:.2f}%")
    
    # Sauvegarder le meilleur
    if vacc > best_acc + config.MIN_DELTA:
        best_acc = vacc
        patience = 0
        
        model_to_save = model.module if hasattr(model, 'module') else model
        
        torch.save({
            'epoch': epoch,
            'model': model_to_save.state_dict(),
            'opt': optimizer.state_dict(),
            'best_acc': best_acc,
            'metadata': metadata,
            'hist': hist
        }, config.MODEL_SAVE_PATH)
        
        print(f"  ✅ Sauvegardé (Acc={best_acc:.2f}%)")
    else:
        patience += 1
        print(f"  Patience: {patience}/{config.PATIENCE}")
        
        if patience >= config.PATIENCE:
            print("\n⏹️ Early stopping")
            break
    
    # NOUVEAU : Sauvegarder checkpoint régulier tous les 3 epochs
    if (epoch + 1) % config.SAVE_EVERY_N_EPOCHS == 0:
        checkpoint_path = config.CHECKPOINT_PATH.format(epoch + 1)
        
        model_to_save = model.module if hasattr(model, 'module') else model
        
        torch.save({
            'epoch': epoch,
            'model': model_to_save.state_dict(),
            'opt': optimizer.state_dict(),
            'best_acc': best_acc,
            'metadata': metadata,
            'hist': hist
        }, checkpoint_path)
        
        print(f"  💾 Checkpoint: epoch_{epoch+1}.pth")
    
    gc.collect()
    torch.cuda.empty_cache()

print(f"\n✅ Terminé! Meilleure accuracy: {best_acc:.2f}%")


🏋️ Début de l'entraînement (epoch 1 à 50)...



Epoch 1/50:   0%|          | 0/2279 [00:00<?, ?it/s]


Epoch 1:
  Train: Loss=6.1830, Acc=5.41%
  Test:  Loss=4.3243, Acc=27.71%


  ✅ Sauvegardé (Acc=27.71%)


Epoch 2/50:   0%|          | 0/2279 [00:00<?, ?it/s]

...

  ✅ Sauvegardé (Acc=96.18%)


  💾 Checkpoint: epoch_21.pth


Epoch 22/50:   0%|          | 0/2279 [00:00<?, ?it/s]


Epoch 22:
  Train: Loss=1.4417, Acc=94.78%
  Test:  Loss=1.3234, Acc=96.16%
  Patience: 1/10


Epoch 23/50:   0%|          | 0/2279 [00:00<?, ?it/s]

## 📊 Visualisation

In [ ]:
fig, (ax1,ax2) = plt.subplots(1,2,figsize=(14,5))

eps = range(1, len(hist['train_loss'])+1)

ax1.plot(eps, hist['train_loss'], 'b-', label='Train')
ax1.plot(eps, hist['test_loss'], 'r-', label='Test')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Loss')
ax1.legend()
ax1.grid(True)

ax2.plot(eps, hist['train_acc'], 'b-', label='Train')
ax2.plot(eps, hist['test_acc'], 'r-', label='Test')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Accuracy')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig('curves.png', dpi=150)
plt.show()

print("✅ Courbes sauvegardées")

## 📥 Fichiers Générés

In [ ]:
!ls -lh /kaggle/working/